# 03 — Model Training & Comparison

Train all three model tiers and compare their ranking metrics:
- **Baseline**: Random Forest
- **Neural**: MLP with champion embeddings
- **Advanced**: Transformer over draft sequence


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

from src.data.preprocess import load_processed
from src.features.champion_encoder import ChampionEncoder, DraftStateEncoder
from src.models import baseline as bm
from evaluation.metrics import compute_all

SEED = 42
df = load_processed()
all_ids = sorted(df['champion_id'].unique().tolist())
champ_enc = ChampionEncoder(all_ids)
state_enc = DraftStateEncoder(champ_enc)

rows = df.to_dict('records')
X = state_enc.encode_batch(rows)
y = np.array(champ_enc.encode_many(df['champion_id'].tolist()), dtype=np.int64)
X_tr, X_test, y_tr, y_test = train_test_split(X, y, test_size=0.15, random_state=SEED)
print(f'Train: {X_tr.shape}, Test: {X_test.shape}')

In [ ]:
# Train Random Forest baseline
rf = bm.RandomForestRecommender(num_champions=champ_enc.num_champions)
rf.fit(X_tr, y_tr)
rf_scores = rf.predict_proba(X_test)
rf_metrics = compute_all(y_test, rf_scores)
print('Random Forest:', rf_metrics)

In [ ]:
# Visualise top-5 recommendation accuracy
results = pd.DataFrame([{'model': 'Random Forest', **rf_metrics}])
metrics_to_plot = ['top1_acc', 'top3_acc', 'top5_acc', 'mrr']
plot_df = results.set_index('model')[metrics_to_plot]
plot_df.plot(kind='bar', figsize=(10, 5), title='Model Comparison — Ranking Metrics')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.ylabel('Score')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Save trained model
import pathlib
rf.save(pathlib.Path('../models/rf_recommender.pkl'))
print('Model saved.')